# Análisis Estadístico de Emprendedores - Medellín

**Dataset:** `registro_artesano_y_producto_formado_y_cualificados_en_diseno.csv`

Este notebook responde las siguientes preguntas analíticas:
1. **Densidad emprendedora por barrio** — emprendedores por cada 1000 hab (DANE)
2. **Tipo dominante de emprendimiento** — moda por barrio/zona
3. **Índice de diversificación de negocios** — tipos únicos / total emprendimientos
4. **Perfil demográfico del emprendedor** — edad y sexo por barrio

> **Nota:** La variable `monto_credito` no está disponible en el dataset actual, por lo que el análisis de capacidad financiera se omite en este notebook.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Cargar dataset ──────────────────────────────────────────────────────────
df = pd.read_csv('../Database/registro_artesano_y_producto_formado_y_cualificados_en_diseno.csv')

print('Shape:', df.shape)
print('\nColumnas:', df.columns.tolist())
print('\nPrimeras filas:')
df.head()

Shape: (199, 7)

Columnas: ['edad', 'sexo', 'tipo_de_emprendimiento', 'idea_de_negocio', 'barrio_vereda_ciudadano', 'comuna_ciudadano', 'zona_ciudadano']

Primeras filas:


,edad,sexo,tipo_de_emprendimiento,idea_de_negocio,barrio_vereda_ciudadano,comuna_ciudadano,zona_ciudadano
0,63.0,masculino,artesanal,emprendedor,robledo,7,nor-occidental
1,56.0,masculino,artesanal,emprendedor,bombona,10,centro-oriental
2,74.0,masculino,artesanal,emprendedor,guayabal,15,sur-oriental
3,54.0,femenino,artesanal,emprendedor,america - santa monica #2,12,centro-oriental
4,50.0,femenino,artesanal,emprendedor,america - santa monica #2,12,centro-oriental


In [2]:
# ── Exploración general ──────────────────────────────────────────────────────
print('=== VALORES NULOS ===')
print(df.isnull().sum())
print('\n=== TIPOS DE DATO ===')
print(df.dtypes)
print('\n=== DESCRIPCIÓN NUMÉRICA ===')
df.describe()

=== VALORES NULOS ===
edad                        5
sexo                        0
tipo_de_emprendimiento     23
idea_de_negocio             5
barrio_vereda_ciudadano     2
comuna_ciudadano            2
zona_ciudadano              2
dtype: int64

=== TIPOS DE DATO ===
edad                       float64
sexo                        object
tipo_de_emprendimiento      object
idea_de_negocio             object
barrio_vereda_ciudadano     object
comuna_ciudadano            object
zona_ciudadano              object
dtype: object

=== DESCRIPCIÓN NUMÉRICA ===


,edad
count,194.000000
mean,50.175258
std,11.617398
min,21.000000
25%,42.000000
50%,52.000000
75%,58.000000
max,75.000000


In [3]:
import re

# ── Normalización de texto ───────────────────────────────────────────────────
df['barrio'] = df['barrio_vereda_ciudadano'].str.strip().str.lower()
df['sexo']   = df['sexo'].str.strip().str.lower()
df['zona']   = df['zona_ciudadano'].str.strip().str.lower()

# Normalizar tipo_de_emprendimiento: unificar variantes
tipo_map = {'artesanias': 'artesanal', 'no formal': 'no_formal'}
df['tipo_emprendimiento'] = (
    df['tipo_de_emprendimiento']
    .str.strip().str.lower()
    .replace(tipo_map)
    .fillna('sin_clasificar')
)

# Extraer número de comuna (maneja '07-robledo', '99-otro', '7', nan, etc.)
def extraer_comuna(val):
    if pd.isna(val):
        return 0
    m = re.match(r'^(\d+)', str(val).strip())
    if m:
        n = int(m.group(1))
        return n if n <= 20 else 0  # >20 son corregimientos → 0
    return 0

df['comuna'] = df['comuna_ciudadano'].apply(extraer_comuna)

print('Tipos de emprendimiento únicos:', df['tipo_emprendimiento'].unique())
print('\nDistribución tipo emprendimiento:')
print(df['tipo_emprendimiento'].value_counts())
print('\nDistribución sexo:')
print(df['sexo'].value_counts())
print('\nComunas únicas:', sorted(df['comuna'].unique()))
print('\nBarrios únicos:', df['barrio'].nunique())

Tipos de emprendimiento únicos: ['artesanal' 'basica' 'no_formal' 'sin_clasificar']

Distribución tipo emprendimiento:
tipo_emprendimiento
artesanal         171
sin_clasificar     23
basica              3
no_formal           2
Name: count, dtype: int64

Distribución sexo:
sexo
femenino     143
masculino     56
Name: count, dtype: int64

Comunas únicas: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16)]

Barrios únicos: 97


## 1. Densidad Emprendedora por Barrio y Comuna

In [4]:
# ── Conteo por barrio ────────────────────────────────────────────────────────
emprendedores_barrio = (
    df.groupby('barrio')
    .agg(
        n_emprendedores=('barrio', 'count'),
        comuna=('comuna', 'first'),
        zona=('zona', 'first')
    )
    .reset_index()
    .sort_values('n_emprendedores', ascending=False)
)

print('=== TOP 15 BARRIOS POR NÚMERO DE EMPRENDEDORES ===')
print(emprendedores_barrio.head(15).to_string(index=False))
print('\n=== ESTADÍSTICAS GLOBALES ===')
print(f'Total emprendedores: {emprendedores_barrio["n_emprendedores"].sum()}')
print(f'Barrios cubiertos: {len(emprendedores_barrio)}')
print(f'Media emprendedores/barrio: {emprendedores_barrio["n_emprendedores"].mean():.2f}')
print(f'Mediana: {emprendedores_barrio["n_emprendedores"].median():.0f}')
print(f'Desv. estándar: {emprendedores_barrio["n_emprendedores"].std():.2f}')

=== TOP 15 BARRIOS POR NÚMERO DE EMPRENDEDORES ===
               barrio  n_emprendedores  comuna              zona
              robledo               14       7    nor-occidental
                belen               12      16    sur-occidental
                 otro               10       0              otro
             castilla                7       5    nor-occidental
         buenos aires                6       9   centro-oriental
             aranjuez                5       4      nor-oriental
           la america                5      12   centro-oriental
          campovaldes                5       4      nor-oriental
             calasanz                5      12   centro-oriental
             laureles                5      11   centro-oriental
manrique central no.2                5       3      nor-oriental
 san antonio de prado                5       0    sur-occidental
               boston                4       8   centro-oriental
              moravia                3 

In [5]:
# ── Densidad por COMUNA usando población DANE estimada ──────────────────────
# Fuente: DANE Proyecciones de población 2023 - Medellín por comunas
poblacion_dane = {
    1: 121000,   # Popular
    2: 100000,   # Santa Cruz
    3: 151000,   # Manrique
    4: 150000,   # Aranjuez
    5: 136000,   # Castilla
    6: 182000,   # Doce de Octubre
    7: 194000,   # Robledo
    8: 131000,   # Villa Hermosa
    9: 149000,   # Buenos Aires
    10: 69000,   # La Candelaria
    11: 108000,  # Laureles-Estadio
    12: 89000,   # La América
    13: 130000,  # San Javier
    14: 129000,  # El Poblado
    15: 88000,   # Guayabal
    16: 196000,  # Belén
}
nombre_comunas = {
    1: 'Popular', 2: 'Santa Cruz', 3: 'Manrique', 4: 'Aranjuez',
    5: 'Castilla', 6: 'Doce de Octubre', 7: 'Robledo', 8: 'Villa Hermosa',
    9: 'Buenos Aires', 10: 'La Candelaria', 11: 'Laureles-Estadio',
    12: 'La América', 13: 'San Javier', 14: 'El Poblado',
    15: 'Guayabal', 16: 'Belén'
}

emprendedores_comuna = (
    df.groupby('comuna')
    .size()
    .reset_index(name='n_emprendedores')
)
emprendedores_comuna = emprendedores_comuna[emprendedores_comuna['comuna'] > 0]
emprendedores_comuna['nombre_comuna'] = emprendedores_comuna['comuna'].map(nombre_comunas)
emprendedores_comuna['poblacion_estimada'] = emprendedores_comuna['comuna'].map(poblacion_dane)
emprendedores_comuna['densidad_x1000_hab'] = (
    emprendedores_comuna['n_emprendedores'] / emprendedores_comuna['poblacion_estimada'] * 1000
).round(4)
emprendedores_comuna = emprendedores_comuna.sort_values('densidad_x1000_hab', ascending=False)

print('=== DENSIDAD EMPRENDEDORA POR COMUNA (por cada 1000 hab) ===')
print(emprendedores_comuna.to_string(index=False))

=== DENSIDAD EMPRENDEDORA POR COMUNA (por cada 1000 hab) ===
 comuna  n_emprendedores    nombre_comuna  poblacion_estimada  densidad_x1000_hab
     12               26       La América               89000              0.2921
     10               11    La Candelaria               69000              0.1594
     11               14 Laureles-Estadio              108000              0.1296
      8               16    Villa Hermosa              131000              0.1221
      5               13         Castilla              136000              0.0956
      4               14         Aranjuez              150000              0.0933
      7               17          Robledo              194000              0.0876
     16               17            Belén              196000              0.0867
      3               12         Manrique              151000              0.0795
      9               11     Buenos Aires              149000              0.0738
     13                6       San Ja

## 2. Tipo Dominante de Emprendimiento por Barrio

In [6]:
# ── Moda del tipo de emprendimiento por barrio ───────────────────────────────
tipo_dominante_barrio = (
    df.groupby('barrio')['tipo_emprendimiento']
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A')
    .reset_index()
    .rename(columns={'tipo_emprendimiento': 'tipo_dominante'})
)

# Distribución global
dist_global = df['tipo_emprendimiento'].value_counts(normalize=True).mul(100).round(2)
print('=== DISTRIBUCIÓN GLOBAL DE TIPOS DE EMPRENDIMIENTO ===')
print(dist_global)

# Moda por zona
tipo_dominante_zona = (
    df.groupby('zona')['tipo_emprendimiento']
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'N/A')
    .reset_index()
    .rename(columns={'tipo_emprendimiento': 'tipo_dominante'})
)
print('\n=== TIPO DOMINANTE POR ZONA ===')
print(tipo_dominante_zona.to_string(index=False))

# Tabla cruzada barrio x tipo
crosstab_tipo = pd.crosstab(df['barrio'], df['tipo_emprendimiento'])
print('\n=== DISTRIBUCIÓN TIPO EMPRENDIMIENTO POR BARRIO ===')
print(crosstab_tipo.head(20))

=== DISTRIBUCIÓN GLOBAL DE TIPOS DE EMPRENDIMIENTO ===
tipo_emprendimiento
artesanal         85.93
sin_clasificar    11.56
basica             1.51
no_formal          1.01
Name: proportion, dtype: float64

=== TIPO DOMINANTE POR ZONA ===
             zona tipo_dominante
centro-occidental      artesanal
  centro-oriental      artesanal
   nor-occidental      artesanal
     nor-oriental      artesanal
        occidente      artesanal
          oriente      artesanal
             otro      artesanal
   sur-occidental      artesanal
     sur-oriental      artesanal

=== DISTRIBUCIÓN TIPO EMPRENDIMIENTO POR BARRIO ===
tipo_emprendimiento                   artesanal  basica  no_formal  \
barrio                                                               
america - santa monica #2                     2       0          0   
aranjuez                                      4       0          1   
barrio las palmas                             1       0          0   
belen                         

## 3. Índice de Diversificación de Negocios

In [7]:
# ── Índice de diversificación: tipos únicos / total emprendimientos ──────────
diversificacion = (
    df.groupby('barrio')
    .agg(
        tipos_unicos=('tipo_emprendimiento', 'nunique'),
        total_emprendedores=('tipo_emprendimiento', 'count'),
        tipos_lista=('tipo_emprendimiento', lambda x: ', '.join(x.unique()))
    )
    .reset_index()
)
diversificacion['indice_diversificacion'] = (
    diversificacion['tipos_unicos'] / diversificacion['total_emprendedores']
).round(4)

# Clasificar: mono-sector vs ecosistema mixto
diversificacion['clasificacion'] = diversificacion['tipos_unicos'].apply(
    lambda x: 'mono-sector' if x == 1 else 'mixto'
)

diversificacion = diversificacion.sort_values('indice_diversificacion', ascending=False)

print('=== ÍNDICE DE DIVERSIFICACIÓN POR BARRIO ===')
print(diversificacion.to_string(index=False))

print('\n=== RESUMEN ===')
print(f"Barrios mono-sector: {(diversificacion['clasificacion'] == 'mono-sector').sum()}")
print(f"Barrios con ecosistema mixto: {(diversificacion['clasificacion'] == 'mixto').sum()}")
print(f"\nÍndice medio de diversificación: {diversificacion['indice_diversificacion'].mean():.4f}")
print(f"Max diversificación: {diversificacion['indice_diversificacion'].max():.4f} → {diversificacion.loc[diversificacion['indice_diversificacion'].idxmax(), 'barrio']}")

=== ÍNDICE DE DIVERSIFICACIÓN POR BARRIO ===
                                   barrio  tipos_unicos  total_emprendedores                          tipos_lista  indice_diversificacion clasificacion
                        barrio las palmas             1                    1                            artesanal                  1.0000   mono-sector
                             el velodromo             1                    1                               basica                  1.0000   mono-sector
                            belen rosales             1                    1                            artesanal                  1.0000   mono-sector
                                   boyaca             1                    1                       sin_clasificar                  1.0000   mono-sector
                       belen san bernardo             1                    1                            artesanal                  1.0000   mono-sector
               caicedo sector la libertad  

Max diversificación: 1.0000 → barrio las palmas


## 4. Perfil Demográfico del Emprendedor

In [8]:
# ── Distribución de edad ─────────────────────────────────────────────────────
print('=== ESTADÍSTICAS GLOBALES DE EDAD ===')
print(df['edad'].describe())
print(f'\nModa de edad: {df["edad"].mode().iloc[0]}')

# Segmentos de edad
bins = [0, 25, 35, 45, 55, 100]
labels = ['<25', '25-34', '35-44', '45-54', '55+']
df['rango_edad'] = pd.cut(df['edad'], bins=bins, labels=labels, right=False)
print('\n=== DISTRIBUCIÓN POR RANGO DE EDAD ===')
print(df['rango_edad'].value_counts().sort_index())

# Por sexo
print('\n=== DISTRIBUCIÓN POR SEXO ===')
print(df['sexo'].value_counts())
print(df['sexo'].value_counts(normalize=True).mul(100).round(2))

=== ESTADÍSTICAS GLOBALES DE EDAD ===
count    194.000000
mean      50.175258
std       11.617398
min       21.000000
25%       42.000000
50%       52.000000
75%       58.000000
max       75.000000
Name: edad, dtype: float64

Moda de edad: 52.0

=== DISTRIBUCIÓN POR RANGO DE EDAD ===
rango_edad
<25       4
25-34    18
35-44    37
45-54    57
55+      78
Name: count, dtype: int64

=== DISTRIBUCIÓN POR SEXO ===
sexo
femenino     143
masculino     56
Name: count, dtype: int64
sexo
femenino     71.86
masculino    28.14
Name: proportion, dtype: float64


In [9]:
# ── Perfil demográfico por barrio ────────────────────────────────────────────
perfil_demografico = (
    df.groupby('barrio')
    .agg(
        n_emprendedores=('barrio', 'count'),
        edad_media=('edad', 'mean'),
        edad_mediana=('edad', 'median'),
        edad_min=('edad', 'min'),
        edad_max=('edad', 'max'),
        edad_std=('edad', 'std'),
        n_femenino=('sexo', lambda x: (x == 'femenino').sum()),
        n_masculino=('sexo', lambda x: (x == 'masculino').sum())
    )
    .reset_index()
)
perfil_demografico['pct_femenino'] = (
    perfil_demografico['n_femenino'] / perfil_demografico['n_emprendedores'] * 100
).round(1)
perfil_demografico['pct_masculino'] = (
    perfil_demografico['n_masculino'] / perfil_demografico['n_emprendedores'] * 100
).round(1)
perfil_demografico['edad_media'] = perfil_demografico['edad_media'].round(1)

print('=== PERFIL DEMOGRÁFICO POR BARRIO ===')
print(perfil_demografico.to_string(index=False))

=== PERFIL DEMOGRÁFICO POR BARRIO ===
                                   barrio  n_emprendedores  edad_media  edad_mediana  edad_min  edad_max  edad_std  n_femenino  n_masculino  pct_femenino  pct_masculino
                america - santa monica #2                2        52.0          52.0      50.0      54.0  2.828427           2            0         100.0            0.0
                                 aranjuez                5        51.4          56.0      39.0      63.0 11.148991           3            2          60.0           40.0
                        barrio las palmas                1        57.0          57.0      57.0      57.0       NaN           1            0         100.0            0.0
                                    belen               12        51.6          53.0      38.0      61.0  8.656404           9            3          75.0           25.0
                            belen rosales                1        50.0          50.0      50.0      50.0       NaN   

In [10]:
# ── Perfil demográfico por zona ──────────────────────────────────────────────
perfil_zona = (
    df.groupby('zona')
    .agg(
        n_emprendedores=('zona', 'count'),
        edad_media=('edad', 'mean'),
        edad_mediana=('edad', 'median'),
        n_femenino=('sexo', lambda x: (x == 'femenino').sum()),
        n_masculino=('sexo', lambda x: (x == 'masculino').sum())
    )
    .reset_index()
)
perfil_zona['pct_femenino'] = (perfil_zona['n_femenino'] / perfil_zona['n_emprendedores'] * 100).round(1)
perfil_zona['edad_media'] = perfil_zona['edad_media'].round(1)

print('=== PERFIL DEMOGRÁFICO POR ZONA ===')
print(perfil_zona.to_string(index=False))

=== PERFIL DEMOGRÁFICO POR ZONA ===
             zona  n_emprendedores  edad_media  edad_mediana  n_femenino  n_masculino  pct_femenino
centro-occidental               25        52.1          52.0          22            3          88.0
  centro-oriental               59        54.4          54.0          35           24          59.3
   nor-occidental               35        48.6          51.0          26            9          74.3
     nor-oriental               31        47.5          50.5          21           10          67.7
        occidente                3        46.0          40.0           2            1          66.7
          oriente                5        43.6          39.0           4            1          80.0
             otro               10        45.8          44.0           9            1          90.0
   sur-occidental               24        48.6          53.0          20            4          83.3
     sur-oriental                5        47.8          50.0    

## 5. Exportar Archivos para Dashboard

In [11]:
import os
OUTPUT_DIR = '../Dashboard/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1. Densidad emprendedora por barrio ─────────────────────────────────────
emprendedores_barrio.to_csv(f'{OUTPUT_DIR}/densidad_emprendedora_barrio.csv', index=False)

# ── 2. Densidad emprendedora por comuna ─────────────────────────────────────
emprendedores_comuna.to_csv(f'{OUTPUT_DIR}/densidad_emprendedora_comuna.csv', index=False)

# ── 3. Tipo dominante por barrio ────────────────────────────────────────────
tipo_dominante_barrio.to_csv(f'{OUTPUT_DIR}/tipo_dominante_barrio.csv', index=False)

# ── 4. Tabla cruzada tipo x barrio ──────────────────────────────────────────
crosstab_tipo.reset_index().to_csv(f'{OUTPUT_DIR}/crosstab_tipo_barrio.csv', index=False)

# ── 5. Índice de diversificación ────────────────────────────────────────────
diversificacion.to_csv(f'{OUTPUT_DIR}/indice_diversificacion_barrio.csv', index=False)

# ── 6. Perfil demográfico por barrio ────────────────────────────────────────
perfil_demografico.to_csv(f'{OUTPUT_DIR}/perfil_demografico_barrio.csv', index=False)

# ── 7. Perfil demográfico por zona ──────────────────────────────────────────
perfil_zona.to_csv(f'{OUTPUT_DIR}/perfil_demografico_zona.csv', index=False)

# ── 8. Tabla consolidada por barrio (master para dashboard) ─────────────────
master_barrio = emprendedores_barrio.copy()
master_barrio = master_barrio.merge(tipo_dominante_barrio, on='barrio', how='left')
master_barrio = master_barrio.merge(
    diversificacion[['barrio','tipos_unicos','indice_diversificacion','clasificacion']],
    on='barrio', how='left'
)
master_barrio = master_barrio.merge(
    perfil_demografico[['barrio','edad_media','pct_femenino','pct_masculino']],
    on='barrio', how='left'
)
master_barrio.to_csv(f'{OUTPUT_DIR}/master_emprendedores_barrio.csv', index=False)

# ── 9. Distribución por rango de edad (global) ───────────────────────────────
dist_edad = df['rango_edad'].value_counts().sort_index().reset_index()
dist_edad.columns = ['rango_edad', 'n_emprendedores']
dist_edad['pct'] = (dist_edad['n_emprendedores'] / dist_edad['n_emprendedores'].sum() * 100).round(2)
dist_edad.to_csv(f'{OUTPUT_DIR}/distribucion_edad_emprendedores.csv', index=False)

print('✅ Archivos exportados exitosamente:')
for f in os.listdir(OUTPUT_DIR):
    print(f'   {OUTPUT_DIR}/{f}')

✅ Archivos exportados exitosamente:
   ../Dashboard/data/crosstab_tipo_barrio.csv
   ../Dashboard/data/densidad_emprendedora_barrio.csv
   ../Dashboard/data/densidad_emprendedora_comuna.csv
   ../Dashboard/data/distribucion_edad_emprendedores.csv
   ../Dashboard/data/indice_diversificacion_barrio.csv
   ../Dashboard/data/master_emprendedores_barrio.csv
   ../Dashboard/data/perfil_demografico_barrio.csv
   ../Dashboard/data/perfil_demografico_zona.csv
   ../Dashboard/data/tipo_dominante_barrio.csv


In [12]:
# ── Resumen estadístico final ────────────────────────────────────────────────
print('=' * 60)
print('RESUMEN ESTADÍSTICO FINAL - EMPRENDEDORES')
print('=' * 60)
print(f'Total registros: {len(df)}')
print(f'Barrios cubiertos: {df["barrio"].nunique()}')
print(f'Comunas cubiertas: {df["comuna"].nunique()}')
print(f'Zonas cubiertas: {df["zona"].nunique()}')
print()
print(f'Tipo de emprendimiento más frecuente: {df["tipo_emprendimiento"].mode().iloc[0]}')
print(f'Barrio con más emprendedores: {emprendedores_barrio.iloc[0]["barrio"]} ({emprendedores_barrio.iloc[0]["n_emprendedores"]})')
print(f'Barrio con mayor densidad (p/ 1000 hab): {emprendedores_comuna.iloc[0]["nombre_comuna"]} ({emprendedores_comuna.iloc[0]["densidad_x1000_hab"]:.4f})')
print()
print(f'Edad media global: {df["edad"].mean():.1f} años')
print(f'Edad mediana global: {df["edad"].median():.0f} años')
print(f'Rango de edad: {df["edad"].min():.0f} - {df["edad"].max():.0f} años')
print()
print(f'% Femenino: {(df["sexo"] == "femenino").mean() * 100:.1f}%')
print(f'% Masculino: {(df["sexo"] == "masculino").mean() * 100:.1f}%')
print()
print(f'Barrios mono-sector: {(diversificacion["clasificacion"] == "mono-sector").sum()}')
print(f'Barrios ecosistema mixto: {(diversificacion["clasificacion"] == "mixto").sum()}')

RESUMEN ESTADÍSTICO FINAL - EMPRENDEDORES
Total registros: 199
Barrios cubiertos: 97
Comunas cubiertas: 17
Zonas cubiertas: 9

Tipo de emprendimiento más frecuente: artesanal
Barrio con más emprendedores: robledo (14)
Barrio con mayor densidad (p/ 1000 hab): La América (0.2921)

Edad media global: 50.2 años
Edad mediana global: 52 años
Rango de edad: 21 - 75 años

% Femenino: 71.9%
% Masculino: 28.1%

Barrios mono-sector: 83
Barrios ecosistema mixto: 14
